# Reddit Author Profiling – Rigorous Experimental Pipeline

**Binary gender classification** with counterfactual ablation, cross-validated evaluation, and statistical significance testing.

### Pipeline Overview
| Step | Description |
|------|-------------|
| 1 | Environment setup & Drive mounting |
| 2 | Download / load raw dataset |
| 3 | Data loading & cleaning |
| 4 | Generate StratifiedGroupKFold splits |
| 5 | Masking: 4 counterfactual text conditions |
| 6 | Masking sparsity statistics |
| 7 | Classical ML cross-validation (4×4 matrix) |
| 8 | Transformer cross-validation (ModernBERT) |
| 9 | LLM zero-shot evaluation |
| 10 | Qualitative feature analysis |

> Each step caches its outputs to Google Drive, so you can restart the runtime without retraining.

## Step 1: Environment Setup & Drive Mounting

Clone the repository (branch `refactor/rigorous-pipeline`), mount Google Drive, and install dependencies.

In [2]:
# Clone the repo (skip if already cloned)
import os

REPO_URL = "https://github.com/lucazini03/nlp-project.git"
REPO_DIR = "nlp-project"
BRANCH = "refactor/rigorous-pipeline"

if not os.path.exists(REPO_DIR):
    !git clone -b {BRANCH} {REPO_URL}
else:
    print(f"Repository already cloned at {REPO_DIR}")
    !cd {REPO_DIR} && git pull origin {BRANCH}

Repository already cloned at nlp-project
From https://github.com/lucazini03/nlp-project
 * branch            refactor/rigorous-pipeline -> FETCH_HEAD
Already up to date.


In [3]:
# Ensure repository source files are importable (adds nlp-project to sys.path)
import os, sys
REPO_DIR = "nlp-project"
# prefer absolute path under current working directory
repo_path = os.path.join(os.getcwd(), REPO_DIR)
if os.path.exists(repo_path):
    abspath = os.path.abspath(repo_path)
    if abspath not in sys.path:
        sys.path.insert(0, abspath)
else:
    # fall back to a relative entry if present
    if os.path.exists(REPO_DIR):
        rel_abspath = os.path.abspath(REPO_DIR)
        if rel_abspath not in sys.path:
            sys.path.insert(0, rel_abspath)
    else:
        print(f"Warning: '{repo_path}' not found. Imports may fail until you set the working directory or run the repo setup cell.")
# show the entry we added
if len(sys.path) > 0:
    print('sys.path[0] =', sys.path[0])

sys.path[0] = /content/nlp-project


In [4]:
# Mount Google Drive (Colab only)
import os, sys

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print("Not running in Colab — Drive mount skipped.")

# Set working directory
REPO_DIR = "nlp-project"
if IN_COLAB and os.path.exists(f"/content/{REPO_DIR}"):
    os.chdir(f"/content/{REPO_DIR}")
elif os.path.exists(REPO_DIR):
    os.chdir(REPO_DIR)
sys.path.insert(0, os.getcwd())

# Configure Drive-backed output paths
if IN_COLAB:
    DRIVE_OUTPUT = "/content/drive/MyDrive/NLP_project"
else:
    DRIVE_OUTPUT = "drive_output"
os.makedirs(DRIVE_OUTPUT, exist_ok=True)

# Patch config.py to use Drive paths
import config
config.DRIVE_ROOT = DRIVE_OUTPUT
config.MODELS_DIR = f"{DRIVE_OUTPUT}/models"
config.RESULTS_DIR = f"{DRIVE_OUTPUT}/results"
config.LLM_CACHE_DIR = f"{DRIVE_OUTPUT}/llm_cache"
os.makedirs(config.MODELS_DIR, exist_ok=True)
os.makedirs(config.RESULTS_DIR, exist_ok=True)
os.makedirs(config.LLM_CACHE_DIR, exist_ok=True)

print(f"Working directory: {os.getcwd()}")
print(f"Drive output root: {DRIVE_OUTPUT}")
print(f"RANDOM_STATE: {config.RANDOM_STATE}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Working directory: /content/nlp-project
Drive output root: /content/drive/MyDrive/NLP_project
RANDOM_STATE: 42


In [5]:
# Install dependencies
!pip install -q spacy scikit-learn tqdm scipy transformers datasets torch google-generativeai openai
!python -m spacy download en_core_web_sm -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 81.2 MB/s eta 0:00:0000:010:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


## Step 2: Download / Load Raw Dataset

Download `gender.csv` (~362 MB) from Google Drive if not already present.

In [6]:
import os
import gdown

#https://drive.google.com/file/d/14poUE5BV9EVm9ofUnp79oHd_AC6SeLpw/view?usp=sharing

DATA_DIR = "data"
DATA_FILE = os.path.join(DATA_DIR, "gender.csv")
GDRIVE_FILE_ID = "14poUE5BV9EVm9ofUnp79oHd_AC6SeLpw"

os.makedirs(DATA_DIR, exist_ok=True)

if not os.path.exists(DATA_FILE):
    print("Downloading dataset...")
    gdown.download(id=GDRIVE_FILE_ID, output=DATA_FILE, quiet=False)
else:
    size_mb = os.path.getsize(DATA_FILE) / 1e6
    print(f"Dataset already exists: {DATA_FILE} ({size_mb:.1f} MB)")

# Quick sanity check
import pandas as pd
df_peek = pd.read_csv(DATA_FILE, nrows=3, engine="python", sep=",",
                       quotechar='"', escapechar="\\", on_bad_lines="skip")
print(f"\nColumns: {list(df_peek.columns)}")
print(f"First 3 rows preview:")
display(df_peek)

Dataset already exists: data/gender.csv (361.8 MB)

Columns: ['auhtor_ID', 'post', 'female']
First 3 rows preview:


,auhtor_ID,post,female
0,t2_rnjzutp,Good on you for being responsible! I know self...,1
1,t2_rnjzutp,"must go to the grocery store with their child,...",1
2,t2_rnjzutp,"things on her videos, and YouTube took the vid...",1


## Step 3: Data Loading & Cleaning

`clean_text()` applies:
1. HTML entity decoding (`html.unescape`)
2. Zero-width / BOM character removal (`\u200b`, `\u200c`, `\u200d`, `\ufeff`)
3. Unicode NFKC normalisation
4. Whitespace collapsing

Then filters out `[deleted]`, `[removed]`, and empty posts (with per-step logging).

In [7]:
!ls .

config.py    prepare_data.py			 report_latex
data	     __pycache__			 requirements.txt
llm_eval.py  qualitative.py			 results
main.py      README.md				 spacy_features.py
masking.py   Reddit_Gender_Classification.ipynb  train_models.py


In [8]:
import logging, pickle, os, config
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(name)s: %(message)s")

import config
from prepare_data import load_and_clean_data

# Check if we already have the masked dataset (which includes cleaning)
masked_data_path = os.path.join(config.DRIVE_ROOT, "masked_data.pkl")

if os.path.exists(masked_data_path):
    print("✓ Masked dataset found on Drive — loading from cache...")
    with open(masked_data_path, "rb") as f:
        df = pickle.load(f)
    print(f"  Loaded {len(df)} rows with columns: {list(df.columns)}")
else:
    print("Loading and cleaning raw data...")
    df = load_and_clean_data()
    print(f"\n✓ Cleaned dataset: {len(df)} rows")

print(f"\nLabel distribution:\n{df['label'].value_counts()}")
print(f"\nSample cleaned text:\n{df['text'].iloc[0][:300]}...")

Loading and cleaning raw data...
Loaded and cleaned 44635 posts  (dropped 0 placeholder, 0 empty)
Label distribution:
label
0    23777
1    20858
Name: count, dtype: int64

✓ Cleaned dataset: 44635 rows

Label distribution:
label
0    23777
1    20858
Name: count, dtype: int64

Sample cleaned text:
Good on you for being responsible! I know self-discipline is difficult, but it's good that you've got your priorities in order. Good luck on your finals. I'm sorry that happened. I think the people who said that genuinely believed it was the best option for you so I can't be too mad at them. I poste...


## Step 4: Generate StratifiedGroupKFold Splits

5-fold cross-validation with:
- **Stratification** on label (balanced male/female per fold)
- **Grouping** on `author_id` (no author in both train and test)
- **Near-duplicate removal** per fold (TF-IDF cosine similarity > 0.95)

In [9]:
import json, os
import numpy as np
from prepare_data import generate_folds
import config

folds_path = os.path.join(config.DRIVE_ROOT, "folds.json")

if os.path.exists(folds_path):
    print("✓ Loading cached fold splits from Drive...")
    with open(folds_path, "r") as f:
        raw = json.load(f)
    folds = [
        {"fold": f["fold"],
         "train_idx": np.array(f["train_idx"]),
         "test_idx": np.array(f["test_idx"])}
        for f in raw
    ]
    for f in folds:
        print(f"  Fold {f['fold']}: train={len(f['train_idx'])}, test={len(f['test_idx'])}")
else:
    print("Generating new fold splits...")
    folds = generate_folds(df)
    # Save to Drive
    serializable = [{"fold": f["fold"],
                     "train_idx": f["train_idx"].tolist(),
                     "test_idx": f["test_idx"].tolist()} for f in folds]
    with open(folds_path, "w") as f:
        json.dump(serializable, f)
    print(f"\n✓ Folds saved to {folds_path}")

# Verify no author overlap in each fold
for f in folds:
    train_authors = set(df.loc[f["train_idx"], "author_id"])
    test_authors = set(df.loc[f["test_idx"], "author_id"])
    overlap = len(train_authors & test_authors)
    print(f"  Fold {f['fold']}: author overlap = {overlap} (should be 0)")

✓ Loading cached fold splits from Drive...
  Fold 0: train=35909, test=8726
  Fold 1: train=36370, test=8265
  Fold 2: train=34731, test=9904
  Fold 3: train=36519, test=8116
  Fold 4: train=35011, test=9624
  Fold 0: author overlap = 0 (should be 0)
  Fold 1: author overlap = 0 (should be 0)
  Fold 2: author overlap = 0 (should be 0)
  Fold 3: author overlap = 0 (should be 0)
  Fold 4: author overlap = 0 (should be 0)


## Step 5: Masking — 4 Counterfactual Text Conditions

All masking is **pure deletion** (no placeholder tokens). Generated from a single spaCy POS pass:

| Condition | What is deleted |
|-----------|----------------|
| **Original** | Nothing (cleaned text) |
| **Structural** | Only tokens whose lemma ∈ Bolukbasi et al. gender word-pairs |
| **Random** (control) | *k* random non-gender, non-punctuation tokens (k = count from structural) |
| **Topical** | Structural ∪ {NOUN, PROPN} — strict superset |

The Random condition isolates the "loss of information" confound from the "loss of gendered content" effect.

In [ ]:
import pickle, os
from masking import apply_masking_in_chunks
from config import get_rng, RANDOM_STATE, DRIVE_ROOT

masked_data_path = os.path.join(DRIVE_ROOT, "masked_data.pkl")
chunks_cache_dir = os.path.join(DRIVE_ROOT, "masking_chunks")

if "text_structural" in df.columns:
    print("✓ Masked columns already present in DataFrame.")
else:
    print("Generating 4 counterfactual text conditions (single spaCy pass)...")
    print("This may take a while on the full dataset...\n")
    rng = get_rng(RANDOM_STATE)
    df = apply_masking_in_chunks(df, num_chunks=5, cache_dir=chunks_cache_dir, rng=rng)

    # Cache to Drive
    with open(masked_data_path, "wb") as f:
        pickle.dump(df, f)
    print(f"\n✓ Masked dataset saved to {masked_data_path}")

# Show example
idx = 0
print(f"\n--- Example document (index {idx}) ---")
print(f"Original:    {df['text'].iloc[idx][:200]}...")
print(f"Structural:  {df['text_structural'].iloc[idx][:200]}...")
print(f"Random:      {df['text_random'].iloc[idx][:200]}...")
print(f"Topical:     {df['text_topical'].iloc[idx][:200]}...")

Generating 4 counterfactual text conditions (single spaCy pass)...
This may take a while on the full dataset...



Generating 4 text conditions (single spaCy pass):  47%|████▋     | 20991/44635 [51:16<33:51, 11.64it/s]   

: 

## Step 6: Masking Sparsity Statistics

Mean/median tokens deleted per document, broken down by class (Male/Female),
to verify that the structural mask is not wildly imbalanced across classes.

In [ ]:
import os
import pandas as pd
from masking import compute_masking_stats
import config

sparsity_path = os.path.join(config.RESULTS_DIR, "masking_sparsity.csv")

if os.path.exists(sparsity_path):
    print("✓ Loading cached sparsity stats...")
    sparsity_df = pd.read_csv(sparsity_path)
else:
    print("Computing masking sparsity statistics...")
    sparsity_df = compute_masking_stats(df)
    sparsity_df.to_csv(sparsity_path, index=False)
    print(f"\n✓ Saved to {sparsity_path}")

display(sparsity_df.style.format({
    "mean_deleted": "{:.2f}",
    "median_deleted": "{:.1f}",
    "std_deleted": "{:.2f}",
    "mean_ratio": "{:.4f}",
}).set_caption("Masking Sparsity per Condition × Class"))

## Step 7: Classical ML Cross-Validation (4×4 Matrix)

For each fold and each of **NB, LR, SVM**:
- Train on all 4 text conditions
- Evaluate on all 4 text conditions → **4×4 matrix**

Reports:
- Mean ± Std Accuracy & Macro-F1 with 95% CI (t-distribution, df=4)
- Majority-class baseline (`DummyClassifier`)
- **Flip Rate** (% of docs whose prediction changes under masking)
- **Corrected paired t-test** (Nadeau & Bengio, 2003) + Cohen's d

In [ ]:
import os, json
import pandas as pd
from train_models import run_cross_validation
import config

cv_summary_path = os.path.join(config.RESULTS_DIR, "cv_summary.csv")
flip_path = os.path.join(config.RESULTS_DIR, "flip_rates.csv")
sig_path = os.path.join(config.RESULTS_DIR, "significance_tests.json")

if os.path.exists(cv_summary_path):
    print("✓ Loading cached classical CV results from Drive...")
    cv_summary = pd.read_csv(cv_summary_path)
    flip_rates = pd.read_csv(flip_path) if os.path.exists(flip_path) else None
    sig_tests = json.load(open(sig_path)) if os.path.exists(sig_path) else None
    classical_results = {
        "summary": cv_summary,
        "flip_rates": flip_rates,
        "significance_tests": sig_tests,
    }
else:
    print("Running classical model cross-validation...")
    print("This trains 3 models × 4 conditions × 5 folds = 60 pipelines.\n")
    classical_results = run_cross_validation(df, folds, save_models=True)
    cv_summary = classical_results["summary"]
    flip_rates = classical_results["flip_rates"]
    sig_tests = classical_results["significance_tests"]

print("\n✓ Classical CV complete.")

### 7a. Results: Accuracy & Macro-F1 Matrices

In [ ]:
import pandas as pd

# Display 4×4 matrices per model
for model_name in cv_summary[cv_summary["model"] != "Baseline (majority)"]["model"].unique():
    model_df = cv_summary[cv_summary["model"] == model_name]
    print(f"\n{'═' * 70}")
    print(f"  {model_name}")
    print(f"{'═' * 70}")

    # Accuracy pivot
    acc_pivot = model_df.pivot(
        index="train_condition", columns="test_condition",
        values="mean_accuracy"
    ).reindex(index=["original", "structural", "random", "topical"],
              columns=["original", "structural", "random", "topical"])

    std_pivot = model_df.pivot(
        index="train_condition", columns="test_condition",
        values="std_accuracy"
    ).reindex(index=["original", "structural", "random", "topical"],
              columns=["original", "structural", "random", "topical"])

    # Format as "mean ± std"
    formatted = acc_pivot.map(lambda x: f"{x:.4f}") + " ± " + std_pivot.map(lambda x: f"{x:.4f}")
    print("\nAccuracy (Mean ± Std):")
    display(formatted)

    # F1 pivot
    f1_pivot = model_df.pivot(
        index="train_condition", columns="test_condition",
        values="mean_macro_f1"
    ).reindex(index=["original", "structural", "random", "topical"],
              columns=["original", "structural", "random", "topical"])

    f1_std = model_df.pivot(
        index="train_condition", columns="test_condition",
        values="std_macro_f1"
    ).reindex(index=["original", "structural", "random", "topical"],
              columns=["original", "structural", "random", "topical"])

    formatted_f1 = f1_pivot.map(lambda x: f"{x:.4f}") + " ± " + f1_std.map(lambda x: f"{x:.4f}")
    print("\nMacro-F1 (Mean ± Std):")
    display(formatted_f1)

# Baseline
baseline = cv_summary[cv_summary["model"] == "Baseline (majority)"].iloc[0]
print(f"\nBaseline (majority class): "
      f"Acc = {baseline['mean_accuracy']:.4f} ± {baseline['std_accuracy']:.4f}, "
      f"F1 = {baseline['mean_macro_f1']:.4f} ± {baseline['std_macro_f1']:.4f}")

### 7b. Flip Rates & Significance Tests

In [ ]:
# Flip Rates
if flip_rates is not None:
    print("Flip Rates (Train Original → Test Masked):")
    print("Percentage of test documents whose prediction changed.\n")
    display(flip_rates.style.format({
        "mean_flip_rate": "{:.4f}",
        "std_flip_rate": "{:.4f}",
    }).set_caption("Counterfactual Flip Rate (Validity)"))

# Significance Tests
if sig_tests is not None:
    print("\n\nCorrected Paired t-test (Nadeau & Bengio, 2003):")
    print("Compares: Train Original → Test Original  vs  Train Original → Test [Masked]\n")
    sig_rows = []
    for model_name, tests in sig_tests.items():
        for test_name, vals in tests.items():
            sig_rows.append({
                "Model": model_name,
                "Comparison": test_name,
                "t-statistic": vals["t_statistic"],
                "p-value": vals["p_value"],
                "Cohen's d": vals["cohens_d"],
                "Mean Δ Accuracy": vals["mean_diff"],
            })
    sig_df = pd.DataFrame(sig_rows)
    display(sig_df.style.format({
        "t-statistic": "{:.4f}",
        "p-value": "{:.6f}",
        "Cohen's d": "{:.4f}",
        "Mean Δ Accuracy": "{:.4f}",
    }).set_caption("Significance Tests"))

## Step 8: Transformer Cross-Validation (ModernBERT)

> ⚠️ **Requires GPU** — set Runtime → Change runtime type → T4 GPU.

Fine-tunes `answerdotai/ModernBERT-base` on all 4 text conditions × 5 folds,
producing the same 4×4 evaluation matrix as the classical models.

In [ ]:
import os
import pandas as pd
from train_models import run_transformer_cv
import config

TRANSFORMER_MODEL = "answerdotai/ModernBERT-base"
short_name = TRANSFORMER_MODEL.split("/")[-1]
tr_summary_path = os.path.join(config.RESULTS_DIR, f"cv_summary_{short_name}.csv")

if os.path.exists(tr_summary_path):
    print(f"✓ Loading cached transformer results ({short_name})...")
    tr_summary = pd.read_csv(tr_summary_path)
    transformer_results = {"summary": tr_summary}
else:
    print(f"Training {TRANSFORMER_MODEL} — 4 conditions × 5 folds = 20 training runs")
    print("This will take a while on GPU...\n")
    transformer_results = run_transformer_cv(
        df, folds,
        model_name=TRANSFORMER_MODEL,
        max_length=256,
        epochs=3,
        batch_size=16,
    )
    tr_summary = transformer_results["summary"]

# Display results
for train_c in ["original", "structural", "random", "topical"]:
    subset = tr_summary[tr_summary["train_condition"] == train_c]
    print(f"\nTrain on {train_c}:")
    for _, row in subset.iterrows():
        print(f"  → Test {row['test_condition']:12s}: "
              f"Acc={row['mean_accuracy']:.4f}±{row['std_accuracy']:.4f}  "
              f"F1={row['mean_macro_f1']:.4f}±{row['std_macro_f1']:.4f}")

## Step 9: LLM Zero-Shot Evaluation

> ⚠️ **Requires API key** — set your Gemini or OpenAI key before running.

Classifies a stratified subsample of 200 test documents per fold using a fixed
zero-shot prompt with `temperature=0`. Results are cached to Drive (JSON) so
you're not re-billed on rerun.

The same 200 document indices are used across all 4 text conditions for strict paired comparison.

In [ ]:
# ── Configure your API key here ──
# Option A: Gemini (default)
import google.genai as genai
genai.configure(api_key="YOUR_GEMINI_API_KEY")  # ← Replace with your key

# Option B: OpenAI (uncomment below instead)
# import os
# os.environ["OPENAI_API_KEY"] = "YOUR_OPENAI_KEY"

# ── Run evaluation ──
import os
import pandas as pd
from llm_eval import evaluate_llm, GeminiProvider, OpenAIProvider
import config

# Choose provider
provider = GeminiProvider(model_id="gemini-1.5-flash")
# provider = OpenAIProvider(model_id="gpt-4o-mini")  # Alternative

llm_summary_path = os.path.join(config.RESULTS_DIR,
                                 f"llm_summary_{provider.provider_name}.csv")

if os.path.exists(llm_summary_path):
    print(f"✓ Loading cached LLM results ({provider.model_id})...")
    llm_summary = pd.read_csv(llm_summary_path)
    llm_flip_path = os.path.join(config.RESULTS_DIR,
                                  f"llm_flip_rates_{provider.provider_name}.csv")
    llm_flips = pd.read_csv(llm_flip_path) if os.path.exists(llm_flip_path) else None
else:
    print(f"Running LLM evaluation with {provider.model_id}...")
    llm_results = evaluate_llm(df, folds, provider=provider)
    llm_summary = llm_results["summary"]
    llm_flips = llm_results["flip_rates"]

print("\n--- LLM Results ---")
display(llm_summary.style.format({
    "mean_accuracy": "{:.4f}", "std_accuracy": "{:.4f}",
    "mean_macro_f1": "{:.4f}", "std_macro_f1": "{:.4f}",
}).set_caption(f"LLM Zero-Shot: {provider.model_id}"))

if llm_flips is not None:
    print("\n--- LLM Flip Rates ---")
    display(llm_flips)

## Step 10: Qualitative Feature Analysis

Trains Logistic Regression on each text condition (using the last fold's train/test split)
and compares the top TF-IDF features driving predictions in the original vs. each masked condition.

Saves per-example analysis CSVs showing prediction flips and feature importance shifts.

In [ ]:
import os
from qualitative import analyze_features
import config

qual_path = os.path.join(config.RESULTS_DIR, "structural_analysis.csv")

# Use the last fold for qualitative analysis
last_fold = folds[-1]
train_df_qual = df.iloc[last_fold["train_idx"]]
test_df_qual = df.iloc[last_fold["test_idx"]]

if os.path.exists(qual_path):
    print("✓ Qualitative analysis CSVs already exist on Drive.")
    print(f"  Check {config.RESULTS_DIR}/ for:")
    print("    - structural_analysis.csv")
    print("    - random_analysis.csv")
    print("    - topical_analysis.csv")
    print("    - top_features_original_female.csv / male.csv")
    print("    - top_features_structural_female.csv / male.csv")
else:
    print("Running qualitative feature analysis on last fold...")
    analyses = analyze_features(train_df_qual, test_df_qual)

print("\n✓ Pipeline complete!")
print(f"All results saved to: {config.RESULTS_DIR}/")
print(f"All models saved to:  {config.MODELS_DIR}/")

### 10a. Inspect Qualitative Results

In [ ]:
import pandas as pd
import config, os

# Load and display structural analysis
structural_path = os.path.join(config.RESULTS_DIR, "structural_analysis.csv")
if os.path.exists(structural_path):
    struct_df = pd.read_csv(structural_path)
    print(f"Structural analysis: {len(struct_df)} examples")
    print(f"  Predictions changed: {struct_df['prediction_changed'].mean():.1%}")
    print(f"  Original accuracy:   {struct_df['original_correct'].mean():.2%}")
    print(f"  Masked accuracy:     {struct_df['masked_correct'].mean():.2%}")

    # Show flipped examples
    flipped = struct_df[struct_df["prediction_changed"]].head(5)
    if len(flipped) > 0:
        print("\n--- Example Prediction Flips (Structural Masking) ---")
        display(flipped[["original_text", "masked_text", "true_label",
                         "original_prediction", "masked_prediction",
                         "original_top_features", "masked_top_features"]])

# Display top features comparison
for cond in ["original", "structural"]:
    for direction in ["female", "male"]:
        path = os.path.join(config.RESULTS_DIR, f"top_features_{cond}_{direction}.csv")
        if os.path.exists(path):
            feat_df = pd.read_csv(path)
            print(f"\nTop-20 {direction} features ({cond}):")
            display(feat_df)

## Summary & File Inventory

All output files are persisted on Google Drive under `DRIVE_ROOT`:

| File | Contents |
|------|----------|
| `masked_data.pkl` | Full dataset with all 4 text columns |
| `folds.json` | Fold indices (reproducible splits) |
| `results/cv_summary.csv` | 4×4 Accuracy/F1 matrix (Mean ± Std, 95% CI) |
| `results/flip_rates.csv` | Flip Rate per model × condition |
| `results/significance_tests.json` | Nadeau–Bengio p-values + Cohen's d |
| `results/masking_sparsity.csv` | Tokens deleted per condition × class |
| `results/llm_summary_*.csv` | LLM zero-shot performance |
| `results/llm_flip_rates_*.csv` | LLM Flip Rates |
| `results/*_analysis.csv` | Per-example qualitative analysis |
| `results/top_features_*.csv` | Top-20 LR features per condition |
| `models/*.pkl` | Pickled sklearn pipelines (per fold × condition) |
| `models/transformer_*` | HuggingFace checkpoints |
| `llm_cache/*.json` | Cached LLM API responses |